In [ ]:
#!pip install qiskit qiskit-ibm-runtime qiskit-aer numpy

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

import numpy as np

In [ ]:
def match_operator(n: int, char_size: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the match operator of size
    `n` with characters of `char_size` bits
    """

    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qout = QuantumRegister(n, "o")
    qc = QuantumCircuit(qx, qy, qout)

    for i in range(n * char_size):
        qc.cx(qx[i], qy[i])
        qc.x(qy[i])

    for i in range(0, n):
        ctrls = list(range(i * char_size, i * char_size + char_size))
        qc.mcx([*qy[ctrls]], qout[i])
    return qc


qubits = 4
char_size = 3

match_operator(qubits, char_size).draw()

In [ ]:
def extension_operator(n: int, i: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the extension operator for an array of size `n`
    for `i`-th lambda/L vector
    """

    shift = 2 ** (i - 1)

    qin = QuantumRegister(n, "in")
    qout = QuantumRegister(n, "out")
    qc = QuantumCircuit(qin, qout)

    for j in range(n - shift):
        qc.ccx(qin[j], qin[j + shift], qout[j])

    return qc


qubits = 8
i = 2
qc = extension_operator(qubits, i)
qc.draw()

In [ ]:
def quantum_rotation(n: int, s: int):
    """
    builds a rotation gate on `n` qubits of `i` positions
    """
    
    qr = QuantumRegister(n, "q")
    qc = QuantumCircuit(qr)

    for i in range(1, int(np.ceil(n / 2))):
        qc.swap(qr[i], qr[n - i])

    for j in range(1, int(np.ceil(n / 2))):
        qc.swap(qr[int(np.ceil(s / 2)) - j], qr[int(np.floor(s / 2)) + j])

    return qc.to_gate(label="R_" + str(s) + " ")


qubits = 8
i = 2
qr = []
qr.append(QuantumRegister(qubits, "q"))

qc = QuantumCircuit(*qr)
qc = qc.compose(quantum_rotation(qubits, i)).decompose()
qc.draw()

In [ ]:
def quantum_parametric_rotation(n: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the parametric rotation gate on `n` qubits.
    The amount of control qubits it's equal to ceil(log2(n))
    """

    l = int(np.ceil(np.log2(n)))

    jr = QuantumRegister(l, "j")
    qr = QuantumRegister(n, "q")
    qc = QuantumCircuit(jr, qr)

    for i in range(l):
        qc = qc.compose(quantum_rotation(n, 2**i).control(1), [jr[i], *qr])
    return qc


qubits = 16

qc = quantum_parametric_rotation(4)
qc.draw()

In [ ]:
def quantum_character_rotation(n: int, char_size: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the parametric rotation on characters of
    size `char_size` qubits of `n` positions
    """
    
    l = int(np.ceil(np.log2(n)))

    jr = QuantumRegister(l, "j")
    qr = QuantumRegister(n * char_size, "q")
    qc = QuantumCircuit(jr, qr)

    for i in range(l):
        controlled_rot = quantum_rotation(n, 2**i).control(1)

        for bit in range(char_size):
            lines = [qr[k * char_size + bit] for k in range(n)]
            qc = qc.compose(controlled_rot, [jr[i], *lines])

    return qc


qc = quantum_character_rotation(8, 2)
qc.draw()

In [ ]:
def copy_operator(n: int):
    """
    builds a copy gate on `n` qubits
    """
    
    qin = QuantumRegister(n, "in")
    qout = QuantumRegister(n, "out")
    qc = QuantumCircuit(qin, qout)

    for i in range(n):
        qc.cx(qin[i], qout[i])

    return qc.to_gate(label="COPY")


qubits = 16
qr = []
qr.append(QuantumRegister(qubits, "q"))

qc = QuantumCircuit(*qr)
qc = qc.compose(copy_operator(qubits // 2)).decompose()
qc.draw()

In [ ]:
def or_operator(n: int):
    """
    builds a logical or gate on `n` qubits
    """
    
    qx = QuantumRegister(n, "x")
    qy = QuantumRegister(1, "y")
    qc = QuantumCircuit(qx, qy)

    qc.x(qx)
    qc.mcx(qx, qy)
    qc.x(qx)
    qc.x(qy)

    return qc.to_gate(label="OR")


qubits = 5
qr = []
qr.append(QuantumRegister(qubits, "q"))

qc = QuantumCircuit(qubits)
qc = qc.compose(or_operator(qubits - 1)).decompose()
qc.draw()

In [ ]:
def contr_bitwise_and_operator(n: int):
    """
    builds a controlled and operator on `n` qubits
    """
    
    qc_ctrl = QuantumRegister(1, "c")
    qL = QuantumRegister(n, "L")
    qin = QuantumRegister(n + 1, "qin")
    qout = QuantumRegister(n + 1, "qout")
    qc = QuantumCircuit(qc_ctrl, qL, qin, qout)

    for k in range(n):
        qc.mcx([qc_ctrl[0], qL[k], qin[k]], qout[k])

    return qc.to_gate(label="C-AND")


qubits = 5
qr = []
qr.append(QuantumRegister(qubits, "q"))

qc = QuantumCircuit(15)
qc = qc.compose(contr_bitwise_and_operator(4)).decompose()
qc.draw()

In [ ]:
def FSM(n, char_size) -> QuantumCircuit:
    """
    builds the Fixed Substring Matching circuit for inputs of size `n` and
    characters of `char_size` qubits
    """

    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len
    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qd = QuantumRegister(d_len, "d")
    qD = QuantumRegister(d_len + 1, "D")
    qL = []

    for i in range(L_num):
        qL.append(QuantumRegister(n, "L[" + str(i) + "]"))

    qD = []
    for i in range(d_len + 1):
        qD.append(QuantumRegister(n + 1, "D[" + str(i - 1) + "]"))

    qr = QuantumRegister(1, "r")
    qc = QuantumCircuit(qx, qy, qd, *qL, *qD, qr)

    qc = qc.compose(
        match_operator(n, char_size).to_gate(label="MATCH"), [*qx, *qy, *qL[0]]
    )

    for i in range(L_num - 1):
        qc = qc.compose(
            extension_operator(n, i + 1).to_gate(label=f"EXT_{i}"),
            [*qL[i][:n], *qL[i + 1]],
        )

    for i in range(d_len):

        qc = qc.compose(
            contr_bitwise_and_operator(n), [qd[i], *qL[i], *qD[i], *qD[i + 1]]
        )

        controlled_rot = quantum_rotation(n + 1, 2**i).control(1)
        qc = qc.compose(controlled_rot, [qd[i], *qD[i + 1]])
        qc.x(qd[i])
        controlled_copy = copy_operator(n + 1).control(1)
        qc = qc.compose(controlled_copy, [qd[i], *qD[i], *qD[i + 1]])
        qc.x(qd[i])

    qc = qc.compose(or_operator(n + 1), [*qD[d_len], qr])

    return qc


qc = FSM(8, 2)
print(qc.num_qubits)
qc.draw()

In [ ]:
def SFC(n, char_size) -> QuantumCircuit:
    """
    builds the Shared Fix Substring Check circuit for inputs of size `n` and
    characters of `char_size` qubits
    """

    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len

    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qd = QuantumRegister(d_len, "d")
    qD = QuantumRegister(d_len + 1, "D")
    qL = []

    for i in range(L_num):
        qL.append(QuantumRegister(n, "L[" + str(i) + "]"))

    qD = []
    for i in range(d_len + 1):
        qD.append(QuantumRegister(n + 1, "D[" + str(i - 1) + "]"))

    qr = QuantumRegister(1, "r")
    qc = QuantumCircuit(qx, qy, qd, *qL, *qD, qr)

    qc.x(qD[0])

    qc = qc.compose(FSM(n, char_size))
    return qc


qc = SFC(4, 1)
qc.draw()

In [ ]:
def FPM(n, char_size) -> QuantumCircuit:
    """
    builds the Fixed Prefix Matching operator for inputs of size `n`
    """
    
    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len
    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qd = QuantumRegister(d_len, "d")
    qD = QuantumRegister(d_len + 1, "D")
    qL = []

    for i in range(L_num):
        qL.append(QuantumRegister(n, "L[" + str(i) + "]"))

    qD = []
    for i in range(d_len + 1):
        qD.append(QuantumRegister(n + 1, "D[" + str(i - 1) + "]"))

    qr = QuantumRegister(1, "r")
    qc = QuantumCircuit(qx, qy, qd, *qL, *qD, qr)

    # init
    qc.x(qD[0][0])

    qc = qc.compose(FSM(n, char_size))
    return qc


qc = FPM(4, 2)
qc.draw()

In [ ]:
def grover_operator(n: int):
    """
    builds the diffusion operator from the grover algorithm
    """

    qx = QuantumRegister(n, "x")
    qc = QuantumCircuit(qx)

    qc.h(qx)
    qc.x(qx)

    if n == 1:
        qc.z(qx[0])
    else:
        qc.h(qx[n - 1])
        qc.mcx(qx[list(range(n - 1))], qx[n - 1])
        qc.h(qx[n - 1])

    qc.x(qx)
    qc.h(qx)

    return qc.to_gate(label="DIFFUSION")


In [ ]:
def int_to_bin(num: int, code_len: int) -> str:
    """
    converts `num` in a binary string, with the specified `code_len`
    """

    num = bin(num)[2:]
    num = "0" * (code_len - len(num)) + num
    return num


In [ ]:
def quantum_number_encode(num: int, code_len: int = 0):
    """
    encodes the given `num` into a gate. if specified, `code_len` lets
    you use more qubits than needed for the number specified by `num`
    """
    
    if num == 0:
        return QuantumCircuit(code_len).to_gate(label=" 0 ")

    if code_len < int(np.ceil(np.log2(num))):
        code_len = len(bin(num)[2:])
    
    bin_num = int_to_bin(num, code_len)
    qc = QuantumCircuit(code_len)

    for c, i in enumerate(bin_num):
        if i == "1":
            qc.x(code_len - 1 - c)

    return qc.to_gate(label=" " + str(num) + " ")


In [ ]:
def encode_boolean_string(x: str) -> QuantumCircuit:
    """
    encodes the given boolean string `x` in a quantum circuit
    """
    n = len(x)
    qx = QuantumRegister(n, "x")
    qc = QuantumCircuit(qx)

    for c, i in enumerate(x):
        if i == "1":
            qc.x(qx[c])

    return qc


In [ ]:
def quantum_step_circuit(x: str, y: str, n: int, d: int, char_size: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the QLCS_test on the strings `x` and `y` of length `n`, 
    checking for the common substring of length `d`, with the specified `char_size`
    """
    
    qi_amount = int(np.ceil(np.log2(n)))
    qi = QuantumRegister(qi_amount, "i")
    qj = QuantumRegister(qi_amount, "j")
    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")

    ancilla_amount = n * int(np.ceil(np.log2(n))) + (n + 1) * (
        int(np.ceil(np.log2(n + 1)))
    )

    qa = QuantumRegister(ancilla_amount, "a")
    qd = QuantumRegister(qi_amount, "d")

    qr = QuantumRegister(1, "r")
    qo = QuantumRegister(1, "out")
    c = ClassicalRegister(1, "c")

    qc = QuantumCircuit(qi, qj, qx, qy, qa, qd, qr, qo, c)

    qc = qc.compose(quantum_number_encode(d, qi_amount), qd)
    qc = qc.compose(encode_boolean_string(x), qx)
    qc = qc.compose(encode_boolean_string(y), qy)
    qc.barrier(label="init")

    # search phase
    qc.h(qj)
    qc.x(qo)
    qc.h(qo)

    epochs = int(np.floor(np.pi / 4 * np.sqrt(n)))

    # THIS IS JUST AN IDEA
    # epochs = max(1, int(np.floor(np.pi/4 * np.sqrt(n)/d)))

    for _ in range(epochs):
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qj, *qx]
        )
        qc = qc.compose(
            SFC(n, char_size).to_gate(label="SFC"), [*qx, *qy, *qd, *qa, qr]
        )

        qc.cx(qr, qo)
        qc = qc.compose(
            SFC(n, char_size).inverse().to_gate(label="- SFC"), [*qx, *qy, *qd, *qa, qr]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qj, *qx],
        )

        qc = qc.compose(grover_operator(qi_amount), qj)

    for j in qj:
        qc.measure(j, c)

    qc.barrier(label="SEARCH PHASE")

    qc.h(qi)

    # verification phase
    for _ in range(epochs):
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qj, *qx]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qx]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qy]
        )
        qc = qc.compose(
            FPM(n, char_size).to_gate(label="FPM"), [*qx, *qy, *qd, *qa, qr]
        )

        qc.cx(qr, qo)
        qc = qc.compose(
            FPM(n, char_size).inverse().to_gate(label="- FPM"), [*qx, *qy, *qd, *qa, qr]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qi, *qy],
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qi, *qx],
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qj, *qx],
        )

        qc = qc.compose(grover_operator(qi_amount), qi)

    for i in qi:
        qc.measure(i, c)
    qc.barrier(label="VERIFICATION PHASE")

    # final check
    qc.h(qo)
    qc.x(qo)

    qc = qc.compose(
        quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qj, *qx]
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qx]
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qy]
    )
    qc = qc.compose(FPM(n, char_size).to_gate(label="FPM"), [*qx, *qy, *qd, *qa, qr])
    qc.cx(qr, qo)

    qc = qc.compose(
        FPM(n, char_size).inverse().to_gate(label="- FPM"), [*qx, *qy, *qd, *qa, qr]
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
        [*qi, *qy],
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
        [*qi, *qx],
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
        [*qj, *qx],
    )

    qc.measure(qo, c)
    qc.barrier(label="FINAL CHECK")

    return qc


quantum_step_circuit("0010", "0000", 2, 1, 2).draw()

In [ ]:
def run(qc: QuantumCircuit, shots=1024):
    """
    returns the result of the execution of the `qc` quantum circuit
    """

    sim_backend = AerSimulator(method="matrix_product_state") # the only viable method for circuits of this size with no entanglement
    sim_backend.set_max_qubits(qc.num_qubits) # as big as needed

    pm = generate_preset_pass_manager(backend=sim_backend, optimization_level=2)
    isa_qc = pm.run(qc)
    sampler = Sampler(mode=sim_backend)

    job = sampler.run([isa_qc], shots=shots)
    results = job.result()
    data = results[0].data
    counts = data.c.get_counts()

    return counts

In [ ]:
def quantum_LCS_test(x: str, y: str, n: int, d: int, char_size: int, shots: int) -> bool:
    """
    returns the result of the QLCS-test on string `x`, `y` of size `n` for substring of length `d`
    on characters of size `char_size`. Runs the circuit for `shots` times (more shots => less false negatives)
    """

    # trivial cases
    if d == 0:
        return True
    if d == n:
        return x == y 

    qc = quantum_step_circuit(x, y, n, d, char_size)
    c = run(qc, shots)

    if "1" in list(c.keys()):
        return True
    return False


In [ ]:
def quantum_LCS(x: str, y: str, n: int, char_size: int, verbose:bool=False, shots:int=32) -> int:
    """
    resolves the longest common substring problem on strings `x`, `y`, 
    of length `n`, with characters of `char_size` bits.
    When `verbose=True`, every test result is printed.
    `shots` lets you specify how many times each circuit is ran.
    """

    l = 0
    r = n
    while l < r:
        d = (l + r + 1) // 2
        if verbose:
            print(f"testing for common substring of length {d}:", end=" ")

        if quantum_LCS_test(x, y, n, d, char_size, shots):
            l = d
            if verbose:
                print("found!")
        else:
            r = d - 1
            if verbose:
                print("not found!")
    return l


In [ ]:
X = "0101010101010101"
Y = "1001101001011010"

print(quantum_LCS(X, Y, 8, 2, True))

In [ ]:
X = "0110011001101111"
Y = "1010011001100000"

print(quantum_LCS(X, Y, 8, 2, True))

In [ ]:
X = "0000000000000000"
Y = "1111111111111111"

print(quantum_LCS(X, Y, 8, 2, True))

# TESTING

In [ ]:
def classical_lcs_substring_len(x: str, y: str, char_size: int = 2) -> int:
    xs = [x[i : i + char_size] for i in range(0, len(x), char_size)]
    ys = [y[i : i + char_size] for i in range(0, len(y), char_size)]

    n = len(xs)
    m = len(ys)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    best = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if xs[i - 1] == ys[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
                best = max(best, dp[i][j])
    return best


tests = [
    # atteso = 4
    ("0101010110000000", "1001010110111111"),
    ("1010101001000000", "0110101010111111"),
    ("0101100110000000", "1001100111111111"),
    ("1010011010000000", "0110011011111111"),
    ("0101100110010000", "1001100110011111"),
    ("1010011001100000", "0110011001101111"),
    ("0101001011010000", "1001001011011111"),
    ("1001010110100000", "0110100110101111"),
    ("0110011010010000", "1001100010011111"),
    ("0010100110100000", "1101010010101111"),
    ("1001101010010000", "0110010010011111"),
    ("0100101011010000", "1011010011011111"),
    ("0010101001010000", "0010101001011111"),
    ("0101010010100000", "0101010010101111"),
    ("1001101001010000", "1001101001011111"),
    ("0110010101100000", "0110010101101111"),
    ("1001100110100000", "1001100110101111"),
    ("0101101001100000", "0101101001101111"),
    ("1010010110100000", "1010010110101111"),
    ("0010100101100000", "0010100101101111"),
    ("0101011001010000", "0101011001011111"),
    ("1001010010100000", "1001010010101111"),
    ("0110011001010000", "0110011001011111"),
    ("1010101001100000", "1010101001101111"),
    ("0010010110100000", "0010010110101111"),
    ("0101100110010000", "0101100110011111"),
    ("1001011001100000", "1001011001101111"),
    ("0110100110010000", "0110100110011111"),
    ("1010011001010000", "1010011001011111"),
    ("1001010110011000", "1001010110011011"),
    ("1001100101100100", "1001100101100111"),
    ("1001100110011000", "1001100110011011"),
    ("1010010101100000", "1010010101101111"),
    ("0110101001010000", "0110101001011111"),
    ("1001100110100000", "1001100110101111"),
    ("0101101001100000", "0101101001101111"),
    ("0101100110010000", "0101100110011111"),
    ("1001011001100000", "1001011001101111"),
    ("0110100110010000", "0110100110011111"),
    ("1010011001010000", "1010011001011111"),
    ("0010101001010000", "0010101001011111"),
    ("0101010010100000", "0101010010101111"),
    ("1001101001010000", "1001101001011111"),
    ("0110010101100000", "0110010101101111"),
    ("1010100110010000", "1010100110011111"),
    ("0010011001100000", "0010011001101111"),
]

passed = 0
failed = 0

for X, Y in tests:
    expected = classical_lcs_substring_len(X, Y, 2)
    print(f"X = '{X}'")
    print(f"Y = '{Y}'")
    actual = quantum_LCS(X, Y, 8, 2, False)
    print("atteso =", expected, "| ottenuto =", actual)

    if expected <= actual:
        passed += 1
    else:
        failed += 1

print(passed)
print(failed)

In [ ]:
tests_3bit = [
    ("001010011100101110000000", "001010011100101110111111"),
    ("010011100101110001010000", "010011100101110001010111"),
    ("011100101110001010000000", "011100101110001010111111"),
    ("100101110001010011100000", "100101110001010011100111"),
    ("101110001010011100000000", "101110001010011100111111"),
    ("110001010011100101000000", "110001010011100101111111"),
    ("001011101010100110000000", "001011101010100110111111"),
    ("010100110101001011000000", "010100110101001011111111"),
    ("011001101100010101000000", "011001101100010101111111"),
    ("100010101001011110000000", "100010101001011110111111"),
    ("101001011010100110000000", "101001011010100110111111"),
    ("110100101001101010000000", "110100101001101010111111"),
    ("001100101010110001000000", "001100101010110001111111"),
    ("010101001100110010000000", "010101001100110010111111"),
    ("011110100101001100000000", "011110100101001100111111"),
    ("100001011010110100000000", "100001011010110100111111"),
    ("101010100110001011000000", "101010100110001011111111"),
    ("110011001010100101000000", "110011001010100101111111"),
    ("001101010011001100000000", "001101010011001100111111"),
    ("010110001101010100000000", "010110001101010100111111"),
]

for X, Y in tests_3bit:
    print(f"X = '{X}'")
    print(f"Y = '{Y}'")
    expected = classical_lcs_substring_len(X, Y, 3)
    actual = quantum_LCS(X, Y, 8, 3, False)
    print("atteso =", expected, "| ottenuto =", actual)

    if expected == actual:
        passed += 1
    else:
        failed += 1

print(passed)
print(failed)

In [ ]:
print(passed)